In [10]:
import json 
import os  

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "greenberg2010chimpanzee")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "Greenberg et al 2010 data.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [11]:
import pandas as pd
import numpy as np
import pyreadstat

df = pd.read_csv(complete_path_1)

df['study_id']="greenberg2010chimpanzee"
df.columns = map(str.lower, df.columns)
df=df.applymap(lambda s: s.lower() if type(s) == str else s)

In [12]:
df.rename(columns={"subject a": "ape", 
    "subject a side":"focal_participant_a_side",
    "subject b":"ape_2",
    "subject b side":"focal_participant_b_side",
    "pull to middle position":"pull_to_middle_position",
    "pull to end position":"pull_to_end_position"}, inplace=True)
df['role'] = "focal_participant_a"
df['role_2'] = "focal_participant_b"
# df.columns

In [13]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
df['ape'] = df['ape'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    df['ape'].replace(x, y, inplace=True)
    df['ape_2'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)   
df= df.merge(apedf,left_on='ape', right_on='name', how='left') 

comp_path_ape_info_2 = os.path.join(pathway_gen, "apes_includeindatabase_2.csv")
apedf_2 = pd.read_csv(comp_path_ape_info_2)
df= df.merge(apedf_2,left_on='ape_2', right_on='name_2', how='left')

df['dyad']=df.ape.str.cat(df.ape_2, sep='_')

In [14]:
df.replace('na', np.nan, inplace=True)
# df.columns
df.rename(columns={"ape": "participant", "ape_2":"participant_2"}, inplace=True)

df['condition'].replace(' ', '_', inplace=True, regex=True)

In [15]:
complete_path_age = os.path.join(original_data_pathway, "subject_list.csv")
subject_list = pd.read_csv(complete_path_age)   
df= df.merge(subject_list,left_on='participant', right_on='name', how='left')
df.rename(columns={"age": "age_in_years"}, inplace=True)

complete_path_age = os.path.join(original_data_pathway, "subject_list_2.csv")
subject_list = pd.read_csv(complete_path_age)   
df= df.merge(subject_list,left_on='participant_2', right_on='name_2', how='left')
df.rename(columns={"age_2": "age_in_years_2"}, inplace=True)

In [16]:
df['helper'].replace('trudi', 'gertrudia', inplace=True, regex=True)

# df['helper'].unique()

In [17]:
greenberg2010chimpanzee_standardized=df[['study_id', 'participant','age_in_years','sex',  'role',
                                         'participant_2','age_in_years_2', 'sex_2','role_2','dyad','species',
        'session','condition','block', 'focal_participant_a_side', 'focal_participant_b_side',
        'helper',  'pull_to_middle_position',
        'pull_to_end_position' ]]


In [18]:
comp_out_path_stand = os.path.join(out_pathway, 'greenberg2010chimpanzee_standardized.csv')
greenberg2010chimpanzee_standardized.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)


names =greenberg2010chimpanzee_standardized.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
greenberg2010chimpanzee_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_pathway, 'greenberg2010chimpanzee_glossary.csv')
greenberg2010chimpanzee_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)
